# Q2: Image Colourization using CNNs

All experiment logs are available on [Weights & Biases](https://wandb.ai/sudhujoshi-smai/smai-a4-q2/workspace?nw=nwusersudhujoshi).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import shutil
import hashlib
import copy
from tqdm import tqdm
import wandb

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10

# 1. Setup Device
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Using GPU: {gpu_name}")
else:
    DEVICE = torch.device("cpu")
    print("No GPU found. Using CPU.")
print(f"Using device object: {DEVICE}")

# 2. Setup Artifacts Directory
artifacts_dir = 'runs-q2'
if os.path.exists(artifacts_dir):
    print(f"Removing old artifacts directory: {artifacts_dir}")
    shutil.rmtree(artifacts_dir)
print(f"Creating clean artifacts directory: {artifacts_dir}")
os.makedirs(artifacts_dir)

# 3. Setup WandB
wandb_dir = 'wandb'
PROJECT_NAME = 'smai-a4-q2'
# Clean up previous wandb runs
if os.path.exists(wandb_dir):
    print(f"Removing old wandb directory: {wandb_dir}")
    shutil.rmtree(wandb_dir)
    
wandb.login()

# 4. Fix Seed for Reproducibility
username = "sudhanva.joshi" 
seed = int(hashlib.sha256(username.encode()).hexdigest(), 16) % (2**32)
print("Seed generated:", seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
CLASS_NAMES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer', 
    'dog', 'frog', 'horse', 'ship', 'truck'
]

PyTorch version: 2.5.1+cu121
Using GPU: NVIDIA GeForce GTX 1650
Using device object: cuda
Removing old artifacts directory: runs-q2
Creating clean artifacts directory: runs-q2
Removing old wandb directory: wandb


wandb: Currently logged in as: sudhujoshi (sudhujoshi-smai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Seed generated: 1027714457


## Colour Centroids and Data Preparation

In [2]:
# 5. Load Color Centroids
try:
    CENTROIDS = np.load('data/color_centroids.npy')
    # Scale centroids to [0, 1] if they are in [0, 255] range
    if CENTROIDS.max() > 1.0:
        print("Scaling centroids from [0, 255] to [0, 1]")
        CENTROIDS = CENTROIDS / 255.0
    else:
        print("Centroids are already in [0, 1] range.")
    CENTROIDS_TENSOR = torch.from_numpy(CENTROIDS).float().to(DEVICE)
    print(f"Loaded color_centroids.npy with shape: {CENTROIDS.shape}")
except FileNotFoundError:
    print("Error: 'color_centroids.npy' not found.")
    CENTROIDS = None
    CENTROIDS_TENSOR = None

def map_image_to_centroids(image_tensor, centroids):
    """
    Maps each pixel of a color image tensor to the nearest centroid index.
    
    Args:
        image_tensor (torch.Tensor): A [3, H, W] color image tensor (values in [0, 1]).
        centroids (torch.Tensor): A [K, 3] tensor of K centroids (values in [0, 1]).
        
    Returns:
        torch.Tensor: A [H, W] tensor of centroid indices (labels).
    """
    H, W = image_tensor.shape[1], image_tensor.shape[2]
    
    # Reshape image to [H*W, 3] and centroids to [K, 3]
    img_flat = image_tensor.permute(1, 2, 0).reshape(-1, 3)
    centroids_flat = centroids.reshape(-1, 3)
    
    # Compute pairwise L2 distance: [H*W, K]
    # (x-y)^2 = x^2 - 2xy + y^2
    dists = (
        torch.sum(img_flat**2, dim=1, keepdim=True)
        - 2 * torch.matmul(img_flat, centroids_flat.T)
        + torch.sum(centroids_flat**2, dim=1).unsqueeze(0)
    )
    
    # Get the index of the minimum distance for each pixel: [H*W]
    labels = torch.argmin(dists, dim=1)
    
    # Reshape back to [H, W]
    return labels.reshape(H, W).long()

class CifarColorizationDataset(Dataset):
    """
    Custom Dataset for CIFAR-10 Colorization.
    Returns (grayscale_image, target_label_map, original_color_image, label)
    """
    def __init__(self, dataset, centroids_tensor, transform_gray=None, transform_color=None):
        self.dataset = dataset
        self.centroids_tensor = centroids_tensor.cpu() # Map on CPU to save GPU memory
        self.transform_gray = transform_gray
        self.transform_color = transform_color if transform_color else transforms.ToTensor()

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # self.dataset returns (PIL Image, label)
        pil_image, label = self.dataset[idx]
        
        # 1. Create grayscale input tensor (normalized)
        gray_image = self.transform_gray(pil_image)
        
        # 2. Create original color tensor (for mapping and visualization)
        color_image_tensor = self.transform_color(pil_image)
        
        # 3. Create target label map
        target_map = map_image_to_centroids(color_image_tensor, self.centroids_tensor)
        
        return gray_image, target_map, color_image_tensor, label

def get_color_dataloaders(batch_size, centroids_numpy, train_val_split_ratio=0.9):
    """
    Loads CIFAR-10, calculates grayscale mean/std, applies transforms,
    and returns train, val, and test dataloaders.
    """
    
    if centroids_numpy is None:
        raise ValueError("Centroids are not loaded. Cannot create dataloaders.")
        
    centroids_tensor = torch.from_numpy(centroids_numpy).float()
    
    # 1. Load data to calculate grayscale mean/std
    temp_transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor()
    ])
    temp_train_data = CIFAR10(root='./data', train=True, download=True, transform=temp_transform)
    
    print("Calculating grayscale dataset mean and std.")
    # Stack all images to compute mean and std
    # Use a subset for faster calculation if necessary, but 50k is manageable
    data_loader = DataLoader(temp_train_data, batch_size=512, shuffle=False, num_workers=2)
    
    mean = 0.0
    std = 0.0
    nb_samples = 0
    for data, _ in tqdm(data_loader, desc="Calc Mean/Std"):
        batch_samples = data.size(0)
        data = data.view(batch_samples, data.size(1), -1)
        mean += data.mean(2).sum(0)
        std += data.std(2).sum(0)
        nb_samples += batch_samples

    mean /= nb_samples
    std /= nb_samples
    print(f"Calculated Grayscale Mean: {mean.item()}, Std: {std.item()}")

    # 2. Define Transforms
    transform_gray_train = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((mean.item(),), (std.item(),))
    ])
    
    transform_gray_val_test = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Normalize((mean.item(),), (std.item(),))
    ])

    transform_color = transforms.ToTensor() # For target mapping and visualization

    # 3. Create Datasets
    # Load full train data (as PIL images)
    full_train_dataset_pil = CIFAR10(root='./data', train=True, download=True)
    
    # Split train/val
    n_train = int(len(full_train_dataset_pil) * train_val_split_ratio)
    n_val = len(full_train_dataset_pil) - n_train
    
    train_subset, val_subset = random_split(full_train_dataset_pil, [n_train, n_val])
    
    # Create custom datasets
    train_data = CifarColorizationDataset(
        train_subset, 
        centroids_tensor, 
        transform_gray=transform_gray_train,
        transform_color=transform_color
    )
    val_data = CifarColorizationDataset(
        val_subset, 
        centroids_tensor, 
        transform_gray=transform_gray_val_test,
        transform_color=transform_color
    )
    
    # Create test dataset
    test_data_pil = CIFAR10(root='./data', train=False, download=True)
    test_data = CifarColorizationDataset(
        test_data_pil,
        centroids_tensor,
        transform_gray=transform_gray_val_test,
        transform_color=transform_color
    )
    
    print(f"Train samples: {len(train_data)}")
    print(f"Validation samples: {len(val_data)}")
    print(f"Test samples: {len(test_data)}")


    # 4. Create DataLoaders
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    
    return train_loader, val_loader, test_loader, mean.item(), std.item()

Scaling centroids from [0, 255] to [0, 1]
Loaded color_centroids.npy with shape: (24, 3)


## Model Architecture

I used ***UNet*** as my model to compare against the given baseline architecture, given it is intended for image classification. The skip connections from encoder to decoder apart from existing botteneck forwards features which have not been dimensionally-reduced.

This leads to significantly better accuracy as the distinguishment task becomes easier for the model to handle.

In [3]:
class BaselineColorizationNet(nn.Module):
    def __init__(self, nic=1, nf=32, nc=24, kernel_size=3):
        super(BaselineColorizationNet, self).__init__()
        
        # Padding to maintain spatial dimensions before pooling
        padding = (kernel_size - 1) // 2
        
        # Encoder
        # Input: [B, 1, 32, 32]
        self.enc_block1 = nn.Sequential(
            nn.Conv2d(nic, nf, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm2d(nf),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # -> [B, NF, 16, 16]
        )
        
        self.enc_block2 = nn.Sequential(
            nn.Conv2d(nf, nf * 2, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm2d(nf * 2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # -> [B, 2*NF, 8, 8]
        )
        
        self.enc_block3 = nn.Sequential(
            nn.Conv2d(nf * 2, nf * 4, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm2d(nf * 4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # -> [B, 4*NF, 4, 4]
        )
        
        # Decoder
        self.dec_block1 = nn.Sequential(
            nn.ConvTranspose2d(nf * 4, nf * 2, kernel_size=2, stride=2),
            nn.BatchNorm2d(nf * 2),
            nn.ReLU() # -> [B, 2*NF, 8, 8]
        )
        
        self.dec_block2 = nn.Sequential(
            nn.ConvTranspose2d(nf * 2, nf, kernel_size=2, stride=2),
            nn.BatchNorm2d(nf),
            nn.ReLU() # -> [B, NF, 16, 16]
        )
        
        self.dec_block3 = nn.Sequential(
            nn.ConvTranspose2d(nf, nc, kernel_size=2, stride=2),
            nn.BatchNorm2d(nc),
            nn.ReLU() # -> [B, NC, 32, 32]
        )
        
        # 1x1 Classifier Conv
        self.classifier = nn.Conv2d(nc, nc, kernel_size=1) # -> [B, NC, 32, 32]

    def forward(self, x):
        # Encoder
        x = self.enc_block1(x)
        x = self.enc_block2(x)
        x = self.enc_block3(x)
        
        # Decoder
        x = self.dec_block1(x)
        x = self.dec_block2(x)
        x = self.dec_block3(x)
        
        # Classifier
        x = self.classifier(x)
        
        return x


# Added UNet implementation with helper blocks
class ConvBlock(nn.Module):
    """(Conv -> BN -> ReLU) * 2"""
    def __init__(self, in_ch, out_ch, kernel_size=3):
        super(ConvBlock, self).__init__()
        padding = (kernel_size - 1) // 2
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=kernel_size, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=kernel_size, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UpConv(nn.Module):
    """Up-sampling (ConvTranspose) followed by ConvBlock"""
    def __init__(self, in_ch, out_ch, kernel_size=3):
        super(UpConv, self).__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = ConvBlock(in_ch, out_ch, kernel_size=kernel_size)

    def forward(self, x1, x2):
        """ x1 is from decoder, x2 is skip connection from encoder """
        x1 = self.up(x1)
        
        # Handle potential size mismatch (e.g., from pooling)
        # This isn't strictly necessary for 32->16->8->4 but is good practice
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = nn.functional.pad(x1, [diffX // 2, diffX - diffX // 2,
                                    diffY // 2, diffY - diffY // 2])
        
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, nic=1, nf=32, nc=24, kernel_size=3):
        super(UNet, self).__init__()
        
        # Encoder (Downsampling path)
        self.inc = ConvBlock(nic, nf, kernel_size=kernel_size)
        self.down1 = nn.Sequential(
            nn.MaxPool2d(2),
            ConvBlock(nf, nf * 2, kernel_size=kernel_size)
        )
        self.down2 = nn.Sequential(
            nn.MaxPool2d(2),
            ConvBlock(nf * 2, nf * 4, kernel_size=kernel_size)
        )
        self.down3 = nn.Sequential(
            nn.MaxPool2d(2),
            ConvBlock(nf * 4, nf * 8, kernel_size=kernel_size)
        )
        
        # Decoder (Upsampling path)
        # Note: in_ch for UpConv is the *total* channels after concatenation
        self.up1 = UpConv(nf * 8, nf * 4, kernel_size=kernel_size)
        self.up2 = UpConv(nf * 4, nf * 2, kernel_size=kernel_size)
        self.up3 = UpConv(nf * 2, nf, kernel_size=kernel_size)
        
        # Final 1x1 convolution
        self.outc = nn.Conv2d(nf, nc, kernel_size=1)

    def forward(self, x):
        # Encoder
        s1 = self.inc(x)       # -> [B, NF, 32, 32]
        s2 = self.down1(s1)    # -> [B, 2*NF, 16, 16]
        s3 = self.down2(s2)    # -> [B, 4*NF, 8, 8]
        b = self.down3(s3)     # -> [B, 8*NF, 4, 4] (Bottleneck)

        # Decoder with skip connections
        x = self.up1(b, s3)    # -> [B, 4*NF, 8, 8]
        x = self.up2(x, s2)    # -> [B, 2*NF, 16, 16]
        x = self.up3(x, s1)    # -> [B, NF, 32, 32]
        
        # Final output
        logits = self.outc(x)  # -> [B, NC, 32, 32]
        return logits

## Training and Evaluation

All training sweeps and results (including images from each class) are given in WandB.

In [4]:
def compute_pixel_accuracy(logits, targets):
    """
    Computes pixel-wise accuracy for colorization.
    
    Args:
        logits: [B, num_classes, H, W] tensor (raw model outputs)
        targets: [B, H, W] tensor (ground truth centroid indices)
        
    Returns:
        float: pixel accuracy (0-100)
    """
    with torch.no_grad():
        preds = torch.argmax(logits, dim=1)   # [B, H, W]
        correct = (preds == targets).sum().item()
        total = targets.numel()
        return 100.0 * correct / total


def train_one_epoch(model, loader, optimizer, criterion, device, epoch):
    model.train()
    total_loss_epoch = 0
    total_acc_epoch = 0

    progress_bar = tqdm(loader, desc=f"Epoch {epoch+1} Train", leave=False)
    for i, (gray_imgs, target_maps, _, _) in enumerate(progress_bar):
        gray_imgs = gray_imgs.to(device)
        target_maps = target_maps.to(device)
        
        # Forward pass
        logits = model(gray_imgs)
        loss = criterion(logits, target_maps)
        
        # Backward + Optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Compute metrics
        batch_loss = loss.item()
        batch_acc = compute_pixel_accuracy(logits, target_maps)
        
        total_loss_epoch += batch_loss
        total_acc_epoch += batch_acc
        
        # Log batch-level metrics
        wandb.log({
            "train/batch/total_loss": batch_loss,
            "train/batch/accuracy": batch_acc,
            "epoch": epoch + 1
        })
        
        progress_bar.set_postfix(loss=batch_loss, acc=f"{batch_acc:.2f}%")

    # Epoch averages
    avg_epoch_loss = total_loss_epoch / len(loader)
    avg_epoch_acc = total_acc_epoch / len(loader)
    
    return avg_epoch_loss, avg_epoch_acc


def validate_one_epoch(model, loader, criterion, device, epoch):
    model.eval()
    total_val_loss = 0
    total_val_acc = 0

    progress_bar = tqdm(loader, desc=f"Epoch {epoch+1} Val", leave=False)
    with torch.no_grad():
        for i, (gray_imgs, target_maps, _, _) in enumerate(progress_bar):
            gray_imgs = gray_imgs.to(device)
            target_maps = target_maps.to(device)
            
            logits = model(gray_imgs)
            loss = criterion(logits, target_maps)
            
            total_val_loss += loss.item()
            total_val_acc += compute_pixel_accuracy(logits, target_maps)

    avg_val_loss = total_val_loss / len(loader)
    avg_val_acc = total_val_acc / len(loader)
    
    wandb.log({
        "val/total_loss": avg_val_loss,
        "val/accuracy": avg_val_acc,
        "epoch": epoch + 1
    })
    
    return avg_val_loss, avg_val_acc


def unnormalize_gray(tensor, mean, std):
    """Un-normalizes a grayscale tensor."""
    mean = torch.tensor(mean).view(1, -1, 1, 1)
    std = torch.tensor(std).view(1, -1, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1)

def reconstruct_image_from_labels(pred_labels, centroids_tensor):
    """
    Converts a map of labels [B, H, W] to a color image [B, 3, H, W]
    using the provided centroids.
    """
    B, H, W = pred_labels.shape
    centroids = centroids_tensor.cpu()
    
    # Get colors for each label index
    # pred_labels.reshape(-1) -> [B*H*W]
    # colors -> [B*H*W, 3]
    colors = centroids[pred_labels.reshape(-1)]
    
    # Reshape to [B, H, W, 3] and permute to [B, 3, H, W]
    return colors.reshape(B, H, W, 3).permute(0, 3, 1, 2).clamp(0, 1)

def log_colorization_examples_to_summary(model, loader, device, gray_mean, gray_std, num_examples=10):
    """
    Gets one sample per class from the loader, runs the model,
    and logs a grid of [Gray, True, Pred] images to wandb.run.summary.
    """
    if CENTROIDS_TENSOR is None:
        print("Cannot log examples, centroids not loaded.")
        return

    model.eval()
    
    # 1. Collect one sample for each class
    samples_by_class = {}
    gray_list = []
    true_color_list = []
    label_list = []
    
    print("Collecting one sample from each class for visualization.")
    
    for gray_imgs, target_maps, true_color_imgs, labels in loader:
        for i in range(len(labels)):
            label = labels[i].item()
            if label not in samples_by_class:
                samples_by_class[label] = True
                gray_list.append(gray_imgs[i])
                true_color_list.append(true_color_imgs[i])
                label_list.append(label)
            if len(samples_by_class) == 10:
                break
        if len(samples_by_class) == 10:
            break
            
    print(f"Collected {len(samples_by_class)} unique class samples.")
    
    if not gray_list:
        print("No samples collected, skipping visualization.")
        return

    # 2. Stack collected images into a batch
    stacked_gray_imgs = torch.stack(gray_list)
    stacked_true_color_imgs = torch.stack(true_color_list)

    # 3. Run model on the collected batch
    with torch.no_grad():
        pred_logits = model(stacked_gray_imgs.to(device)).cpu()
    
    # 4. Get predicted labels and reconstruct
    pred_labels = torch.argmax(pred_logits, dim=1) # [B, 32, 32]
    pred_color_imgs = reconstruct_image_from_labels(pred_labels, CENTROIDS_TENSOR)
    
    # 5. Un-normalize grayscale images
    gray_unnorm = unnormalize_gray(stacked_gray_imgs, gray_mean, gray_std)
    
    # 6. Create plot
    num_examples = len(gray_list)
    fig, axs = plt.subplots(num_examples, 3, figsize=(8, num_examples * 2.5))
    
    if wandb.run:
        title = f"Final Colorization Examples (Run: {wandb.run.name})"
    else:
        title = "Final Colorization Examples"
    fig.suptitle(title)
    
    for i in range(num_examples):
        class_name = CLASS_NAMES[label_list[i]]
        
        g_img = gray_unnorm[i].permute(1, 2, 0).squeeze()
        t_img = stacked_true_color_imgs[i].permute(1, 2, 0)
        p_img = pred_color_imgs[i].permute(1, 2, 0)
        
        axs[i, 0].imshow(g_img, cmap='gray')
        axs[i, 0].set_title("Grayscale Input")
        axs[i, 0].axis('off')
        
        axs[i, 1].imshow(t_img)
        # Add class name to title
        axs[i, 1].set_title(f"Ground Truth ({class_name})") 
        axs[i, 1].axis('off')
        
        axs[i, 2].imshow(p_img)
        axs[i, 2].set_title("Predicted")
        axs[i, 2].axis('off')
        
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    
    plt.text(0.95, 0.01, username, ha='right', va='bottom',
             transform=fig.transFigure, fontsize=10, color='gray', alpha=0.7)
    
    # Log to wandb.run.summary (like Q1)
    wandb.run.summary["val/colorization_examples"] = wandb.Image(fig)
    plt.close(fig)
    print("Logged final colorization examples to wandb summary.")

## Analytical Questions

This analysis calculates the number of weights (parameters), activations (outputs), and connections (computational cost) in both models for a general $S \times S$ input and specifically for $S=32$ vs $S=64$.

**Assumptions (Both Models):**
* NIC = 1, NC = 24.
* NF is the base filter count (e.g., 16).
* K is the conv kernel size (e.g., 3).
* Biases and BatchNorm parameters are ignored.


### 1. Baseline Analysis

**Formulas:**
* **Weights:** $Weights = K^2(NIC \cdot NF + 10 \cdot NF^2) + 40 \cdot NF^2 + 4 \cdot NF \cdot NC + NC^2$
* **Activations:** $Activations(S) = S^2 \cdot [NIC + 2NC + 2.5625 \cdot NF]$
* **Connections:** $Connections(S) = S^2 \cdot [ K^2(NIC \cdot NF + NF^2) + 4 \cdot NF^2 + 4 \cdot NF \cdot NC + NC^2 ]$

**Conclusions ($S=32 \to S=64$):**
* **Weights:** **No change.** The formula is independent of $S$.
* **Activations:** **$4\times$ increase.** The formula scales with $S^2$. ($64^2 = 4 \times 32^2$).
* **Connections:** **$4\times$ increase.** The formula scales with $S^2$.

### 2. UNet Analysis

**Formulas:**
* **Weights:** $Weights = K^2(NIC \cdot NF + 190 \cdot NF^2) + 42 \cdot NF^2 + NF \cdot NC$
* **Activations:** $Activations(S) = S^2 \cdot [ NIC + NC + 9.4375 \cdot NF ]$
* **Connections:** $Connections(S) = S^2 \cdot [ K^2(NIC \cdot NF + 14.5 \cdot NF^2) + 6 \cdot NF^2 + NF \cdot NC ]$

**Conclusions ($S=32 \to S=64$):**
* **Weights:** **No change.** The formula is independent of $S$.
* **Activations:** **$4\times$ increase.** The formula scales with $S^2$.
* **Connections:** **$4\times$ increase.** The formula scales with $S^2$.

In [5]:
def run_colorization_experiment(config):
    """
    Main function to run a single experiment with a given config.
    """
    # 1. Initialize WandB Run
    run_name = (f"model_{config['model_type']}_nf_{config['nf']}_k_{config['kernel_size']}_lr_{config['lr']:.3g}_"
                f"bs_{config['batch_size']}_opt_{config['optimizer']}")

    wandb.init(
        project=PROJECT_NAME,
        config=config,
        name=run_name,
        reinit=True
    )
    
    print(f"\n Starting Run: {run_name}")
    cfg = wandb.config
    
    # 2. Load Data
    train_loader, val_loader, test_loader, gray_mean, gray_std = get_color_dataloaders(
        batch_size=cfg.batch_size, 
        centroids_numpy=CENTROIDS
    )
    
    # 3. Initialize Model, Optimizer, Loss
    if cfg.model_type == 'unet':
        model = UNet(
            nic=1,
            nf=cfg.nf,
            nc=24,
            kernel_size=cfg.kernel_size
        ).to(DEVICE)
    else: # 'baseline'
        model = BaselineColorizationNet(
            nic=1,
            nf=cfg.nf,
            nc=24,
            kernel_size=cfg.kernel_size
        ).to(DEVICE)
    
    print(f"Instantiated model: {cfg.model_type}")
    
    wandb.watch(model, log='all', log_freq=200)
    
    if cfg.optimizer == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=cfg.lr, momentum=0.9)
    else:
        optimizer = optim.Adam(model.parameters(), lr=cfg.lr)
        
    criterion = nn.CrossEntropyLoss()
    
    # 4. Training Loop
    best_val_loss = float('inf')
    best_model_loss_state = None
    
    for epoch in range(cfg.epochs):
        # Train
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, DEVICE, epoch
        )
        
        wandb.log({
            "train/epoch/total_loss": train_loss,
            "train/epoch/accuracy": train_acc,
            "epoch": epoch + 1
        })
        
        # Validate
        val_loss, val_acc = validate_one_epoch(
            model, val_loader, criterion, DEVICE, epoch
        )
        
        print(f"Epoch {epoch+1}/{cfg.epochs} -> "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        
        # Model Selection (based on val loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_loss_state = copy.deepcopy(model.state_dict())
            wandb.run.summary["best_val_loss"] = best_val_loss
            wandb.run.summary["best_val_acc"] = val_acc
            
            # Save best checkpoint
            model_path = os.path.join(artifacts_dir, f"{wandb.run.id}_best.pth")
            torch.save(best_model_loss_state, model_path)
            
    wandb.run.summary["final_train_loss"] = train_loss
    wandb.run.summary["final_val_loss"] = val_loss
    wandb.run.summary["final_train_accuracy"] = train_acc
    wandb.run.summary["final_val_accuracy"] = val_acc

    print(f"Run {run_name} Finished. Best Val Loss: {best_val_loss:.4f}")

    # 6. Final Evaluation on Test Set (Moved before viz, like Q1)
    print("Evaluating on Test Set.")
    if best_model_loss_state is not None:
        model.load_state_dict(best_model_loss_state)
        
        model_name = "best_loss" 
        
        test_loss, test_acc = validate_one_epoch(
            model, test_loader, criterion, DEVICE, epoch=-1 # -1 for test epoch
        )
        
        # Log both values to the summary
        wandb.run.summary[f"test_loss_on_{model_name}_model"] = test_loss
        wandb.run.summary[f"test_accuracy_on_{model_name}_model"] = test_acc
        
        # Print both values
        print(f"Test Loss (using {model_name} model): {test_loss:.4f}, Test Acc: {test_acc:.2f}%")
    
    else:
        print("Skipping test evaluation, no best model was saved.")

    # 7. Log Final Visualizations 
    print("Logging final visualizations.")
    if best_model_loss_state is not None:
        # Model is already loaded with best state
        log_colorization_examples_to_summary(
            model, 
            test_loader, # Use test_loader 
            DEVICE, 
            gray_mean, 
            gray_std,
            num_examples=10
        )
    
    # 8. Finish Run
    wandb.finish()

## Hyperparameter Tuning

Runs show a clear difference in performance between the given baseline and UNet architecture, with UNet showing an accuracy increase of *around* 15%. 
The best run's results can be viewed [here](https://wandb.ai/sudhujoshi-smai/smai-a4-q2/runs/kzkzer2n?nw=nwusersudhujoshi).

In [6]:
sweep_config = {
    'method': 'bayes',  # Bayesian optimization
    'metric': {
        'name': 'val/epoch/loss',
        'goal': 'minimize'
    },
    'parameters': {
        'epochs': {
            'value': 25  # Fixed as per PDF
        },
        'model_type': {
            'values': ['baseline', 'unet']
        },
        'batch_size': {
            'values': [32, 64, 128]
        },
        'lr': {
            'distribution': 'log_uniform_values',
            'min': 1e-4,
            'max': 3e-3
        },
        'optimizer': {
            'values': ['Adam', 'SGD']
        },
        'nf': {
            'values': [8, 16, 32]
        },
        'kernel_size': {
            'values': [3, 5]
        }
    }
}

# 2. Wrapper function for wandb.agent
def sweep_agent_run():
    wandb.init()  # Initialize the run
    config = wandb.config  # Get the config for this run

    run_config = {
        "epochs": config.epochs,
        "batch_size": config.batch_size,
        "optimizer": config.optimizer,
        "nf": config.nf,
        "kernel_size": config.kernel_size,
        "lr": config.lr,
        "model_type": config.model_type
    }
    
    try:
        if CENTROIDS is None:
            print("Skipping run as color_centroids.npy was not loaded.")
            wandb.finish()
            return
        run_colorization_experiment(run_config)
    except Exception as e:
        print(f"Run failed with error: {e}")
        # Optionally log the error to wandb
        wandb.log({"error": str(e)})
        wandb.finish()  # Ensure run is finished even on error


# 3. Initialize the Sweep
sweep_id = wandb.sweep(sweep_config, project=PROJECT_NAME)

# 4. Run Agent (set count to at least 20)
print(f"Running sweep agent with ID: {sweep_id}")
wandb.agent(sweep_id, function=sweep_agent_run, count=20)

print("\nAll sweep experiments completed.")

Create sweep with ID: 12fo90af
Sweep URL: https://wandb.ai/sudhujoshi-smai/smai-a4-q2/sweeps/12fo90af
Running sweep agent with ID: 12fo90af


wandb: Agent Starting Run: pxmwiiib with config:
wandb: 	batch_size: 32
wandb: 	epochs: 25
wandb: 	kernel_size: 3
wandb: 	lr: 0.00012541931845012834
wandb: 	model_type: baseline
wandb: 	nf: 16
wandb: 	optimizer: Adam


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.



 Starting Run: model_baseline_nf_16_k_3_lr_0.000125_bs_32_opt_Adam
Files already downloaded and verified
Calculating grayscale dataset mean and std.


Calc Mean/Std: 100%|██████████| 98/98 [00:04<00:00, 21.09it/s]


Calculated Grayscale Mean: 0.4808611571788788, Std: 0.1963193714618683
Files already downloaded and verified
Files already downloaded and verified
Train samples: 45000
Validation samples: 5000
Test samples: 10000
Instantiated model: baseline


... (633 outputs left out)

Test Loss (using best_loss model): 2.0613, Test Acc: 33.03%
Logging final visualizations.


Collected 10 unique class samples.
Logged final colorization examples to wandb summary.


epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇████
train/batch/accuracy,▃▅▃▅▁▄▆▅▄▄▄▂▆▅▅▅▅▇▅▅▆▄▄▃▆▄▅▆▅▄█▇▃▅▃▇▃▅▄▆
train/batch/total_loss,█▄▂▂▂▂▂▂▂▂▂▃▂▂▂▂▂▂▂▁▂▂▂▂▁▁▂▂▃▂▂▁▁▂▂▂▁▂▂▁
train/epoch/accuracy,▁▅▆▇▇▇▇▇▇▇▇██████████████
train/epoch/total_loss,█▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/accuracy,▁▄▃▅▅▆▅▆▆▆▇▇▇▇▇▇▇▇▇▆▇██▇▆█
val/total_loss,█▅▅▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▂▁
best_val_acc,33.03955
best_val_loss,2.06886
epoch,0
final_train_accuracy,26.70863



All sweep experiments completed.
